# The box, in 3D

The figures beside this notebook are flat: slices, floor plans, a side view. Each one throws an
axis away, and the thing being built is a volume. This is the same window in three dimensions,
where you can spin it.

Everything comes off one checkpoint and one window of the **held-out** building, the same defaults
`figures.py` uses. Plotly is imported here but is not a declared dependency of the package yet —
it arrives via wandb.

**Camera y points down** in this codebase, so every plot below negates it. Up on your screen is up
in the room.


## Setup


In [ ]:
%load_ext autoreload
%autoreload 3
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import plotly.graph_objects as go
import torch

from hexai.data import split_context_target
from hexai.models.vvae.figures import rebuild, window
from hexai.utilities import carve_window

CKPT = Path('../.runs/geo-c32g4-depth-onecycle-b0.0001-lr0.003-jit-unk-lift-g32x60x32c4.pt')
INDEX = 660  # figures.py's default: a wall with boxes on the floor

np.random.seed(42)  # head jitter is drawn per load, so pin which way it was aimed
model, state, cfg = rebuild(CKPT)
batch = window(cfg, 'Office', INDEX)
frames, poses = batch
config = model.config

# after the hexai imports, never before: they set a headless backend for detached runs and
# a notebook that inherits it draws every matplotlib figure into the void
%matplotlib inline
print(f"epoch {state['epoch']}, grid {cfg['grid']}, holding out {cfg['val_env']}")


## What it was given

Seven frames about 0.21 m apart, rgb over depth. This is the whole input — no other sensor, no
prior map, and the poses are noisy at the level the servos actually deliver.


In [ ]:
def show_frames(note='the frames this view was built from'):
    """The window's rgb over its depth. Called above each 3D plot, so what went in is on the
    screen beside what came out rather than scrolled away."""
    n = frames.shape[2]
    fig, axes = plt.subplots(2, n, figsize=(14, 4.2))
    for t in range(n):
        axes[0, t].imshow(frames[0, :3, t].permute(1, 2, 0).numpy())
        axes[1, t].imshow(frames[0, 3, t].numpy(), cmap='magma', vmin=0, vmax=1)
        axes[0, t].set_title(f'frame {t}' + (' (anchor)' if t == n // 2 else ''), fontsize=9)
    for ax in axes.flat: ax.set(xticks=[], yticks=[])
    fig.suptitle(note, fontsize=10)
    fig.tight_layout()
    plt.show()

show_frames()


## Helper: voxel centres in metres

One place that turns a grid index into a position, so every plot below sits in the same room.


In [ ]:
def centres(config):
    """[X, Y, Z, 3] of voxel centres in metres, in the anchor camera's frame."""
    lo, hi = config.bounds[0], config.bounds[1]
    res = torch.tensor(config.grid_resolution)
    axes_m = [torch.linspace(float(lo[i]), float(hi[i]), int(res[i])) for i in range(3)]
    return torch.stack(torch.meshgrid(*axes_m, indexing='ij'), dim=-1)

def scatter(mask, values, config, name, colourscale, size=2.4):
    """Voxels where mask is true, as a 3D scatter. y is negated so up is up."""
    xyz = centres(config)[mask]
    return go.Scatter3d(x=xyz[:, 0], y=xyz[:, 2], z=-xyz[:, 1], mode='markers', name=name,
                        marker=dict(size=size, color=values[mask], colorscale=colourscale,
                                    opacity=0.85, symbol='square'))

SCENE = dict(xaxis_title='x, metres (right)', yaxis_title='z, metres (forward)',
             zaxis_title='height above the lens, metres',
             aspectmode='data', camera=dict(eye=dict(x=1.6, y=-1.9, z=1.0)))
print('grid', config.grid_resolution, 'over', config.bounds.tolist(), 'metres')


## The carving, as a solid

The three states the camera can actually report. **Occupied** is where a ray stopped, **free** is
space a ray crossed to get there, and everything not drawn is **unknown** — behind a surface or
outside the 49° cone.

The unknown is the point. It is most of the box, it is not empty, and a lift alone cannot tell
the two apart. Spin this and the free space is a cone opening away from the camera, with the
furniture's shadow cut out of it.


In [ ]:
show_frames('what the carving below was built from')

carved = carve_window(frames[0], poses[0], config.bounds, config.grid_resolution,
                      config.source_fovy_deg, config.depth_max)
occupied, observed = carved[0] > 0.5, carved[1] > 0.5
free = observed & ~occupied

thin = torch.zeros_like(free, dtype=torch.bool)  # free space is dense, so show a lattice of it
thin[::2, ::2, ::2] = True

fig = go.Figure([
    scatter(free & thin, torch.zeros_like(carved[0]), config, 'free', [[0, '#2a9d8f'], [1, '#2a9d8f']], 1.6),
    scatter(occupied, torch.ones_like(carved[0]), config, 'occupied', [[0, '#e9c46a'], [1, '#e9c46a']], 3.0),
])
fig.update_traces(selector=dict(name='free'), marker_opacity=0.12)
fig.update_layout(scene=SCENE, height=650, title=f'carved: {float(occupied.float().mean()):.1%} occupied, '
                  f'{float(free.float().mean()):.1%} free, {float((~observed).float().mean()):.1%} unknown')
fig


## What the lift built, and what the decoder gives back

Left is exact geometry — every pixel unprojected by its depth and splatted into the box. Right is
the decoder's reconstruction from those 3840 numbers. Same voxel-square style as the carving, so
the two are comparable by eye.

**Drag the threshold.** The lift is nearly binary — a voxel was hit or it was not — so it barely
changes as you slide. The decoder emits a smooth field, so it melts: generous at 0.1, skeletal by
0.7. That difference *is* the blur, and it is why `decoded_solid` has to be read against
`lifted_occupied` at a stated cut rather than quoted on its own.


In [ ]:
show_frames('what both volumes below were built from')

with torch.no_grad():
    context, context_poses, _, query = split_context_target(frames, poses, 1, cfg['target'])
    out = model(context, context_poses, query)
    lifted = model.encoder.lift(context, context_poses)[0, -1]
    decoded = model.decoder.volume(out.mu)[0, config.density_index]

from plotly.subplots import make_subplots
CUTS = [0.1, 0.2, 0.3, 0.5, 0.7]

def solid(density, cut, name, show):
    """Voxels above the cut, as squares coloured by how dense they are."""
    mask = density > cut
    xyz = centres(config)[mask]
    return go.Scatter3d(x=xyz[:, 0], y=xyz[:, 2], z=-xyz[:, 1], mode='markers', name=name,
                        visible=show, showlegend=False,
                        marker=dict(size=2.6, symbol='square', opacity=0.85,
                                    color=density[mask].detach().numpy(),
                                    colorscale='Magma', cmin=0.0, cmax=1.0))

fig = make_subplots(rows=1, cols=2, specs=[[{'type': 'scatter3d'}, {'type': 'scatter3d'}]],
                    subplot_titles=('lifted from depth (exact)', 'decoded from the latent'))
for i, cut in enumerate(CUTS):
    fig.add_trace(solid(lifted, cut, 'lifted', i == 2), row=1, col=1)
    fig.add_trace(solid(decoded, cut, 'decoded', i == 2), row=1, col=2)

steps = []
for i, cut in enumerate(CUTS):
    vis = [j // 2 == i for j in range(2 * len(CUTS))]
    filled = (float((lifted > cut).float().mean()), float((decoded > cut).float().mean()))
    steps.append(dict(method='update', label=f'{cut:.1f}',
                      args=[{'visible': vis},
                            {'title': f'above {cut:.1f}: lifted {filled[0]:.1%} of the box, '
                                      f'decoded {filled[1]:.1%}'}]))
fig.update_layout(scene=SCENE, scene2=SCENE, height=640,
                  sliders=[dict(active=2, currentvalue=dict(prefix='density threshold '),
                                steps=steps)])
fig


## The latent itself

This is the thing that ships. `[4, 8, 15, 8]` — 3840 numbers describing a 4 x 3 x 4 m box, one
latent voxel per 50 x 20 x 50 cm.

The architecture's whole claim is that the latent's **axes are space**, so a latent voxel should
mean something about the region it sits in. Each channel is drawn on the same diverging scale,
centred at zero, since these are signed: red is above the prior, blue below, white is uninformative.

Measured over eight val windows, no channel is dead — the KL splits 19/19/16/46% across the four —
but channel 3 carries roughly twice what the others do.


In [ ]:
show_frames('what this latent encodes')

mu = out.mu[0].detach()
lat = model.config.latent_grid
lo, hi = config.bounds[0], config.bounds[1]
ax_m = [torch.linspace(float(lo[i]), float(hi[i]), lat[i]) for i in range(3)]
lxyz = torch.stack(torch.meshgrid(*ax_m, indexing='ij'), dim=-1)
limit = float(mu.abs().max())

fig = go.Figure()
for c in range(mu.shape[0]):
    v = mu[c]
    fig.add_trace(go.Scatter3d(
        x=lxyz[..., 0].flatten(), y=lxyz[..., 2].flatten(), z=-lxyz[..., 1].flatten(),
        mode='markers', name=f'channel {c}', visible=(c == 0),
        marker=dict(size=6, symbol='square', color=v.flatten(), colorscale='RdBu_r',
                    cmin=-limit, cmax=limit, opacity=0.9, colorbar=dict(title='latent'))))

buttons = [dict(label=f'channel {c}', method='update',
                args=[{'visible': [i == c for i in range(mu.shape[0])]}])
           for c in range(mu.shape[0])]
fig.update_layout(scene=SCENE, height=650, updatemenus=[dict(buttons=buttons, x=0, y=1.1)],
                  title=f'latent {tuple(mu.shape)}, one square per latent voxel')
fig


## Reading it

What to look for, in the order that has actually caught problems:

- **The free cone should open away from the camera** and be interrupted by whatever the depth saw.
  If it is not a cone, the fov or the pose convention is wrong, and no amount of training fixes that.
- **Lifted and decoded should agree where a camera looked.** They mostly do. Where the decoder
  differs is unobserved space, which nothing in the loss constrains.
- **The latent should vary across space.** A channel that reads the same everywhere has stopped
  being a map however well the render scores, and no flat figure shows that as plainly as this one.

The honest state of it: depth error averages about 1.0 m over a 4 m box on the held-out building,
and the decoded cloud is a smeared version of the lifted one rather than a sharper one. The
geometry underneath is verified; the learned half is not there yet.
